# Code Runner Local Exec + academic-search Skill Example

This notebook uses the local non-isolated `code_runner` pattern and explicitly makes the agent use
`skills/academic-search/arxiv_search.py` (and a modified copy of it).

## Prerequisites

1. Install dependency for the skill script
```bash
uv run python -m pip install arxiv
```
2. Prepare non-secret env file for code runner
```bash
cat > .env.code_runner.local <<'ENV'
PYTHONUTF8=1
ENV
```

Note: keys containing `KEY/TOKEN/SECRET/PASSWORD` are blocked in this notebook.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import shlex
import subprocess

from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, FilesystemBackend, StateBackend
from deepagents.backends.protocol import ExecuteResponse, SandboxBackendProtocol
from langchain.tools import ToolRuntime
from langchain_openai import ChatOpenAI

import sys
from pathlib import Path

ROOT = Path("/Users/jungj2/Dropbox/Metadata/aiagent/DeepMAST")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.agents.drug_target.tools import (
    simple_reflection_tool,
)

In [3]:
class GuardedLocalExecBackend(FilesystemBackend, SandboxBackendProtocol):
    """Local non-isolated execute backend with guardrails."""

    ALLOWED_PREFIXES = {"python", "python3", "uv", "pip", "pip3"}
    BLOCKED_TOKENS = {"&&", "||", ";", "|", "`", "$()"}

    def __init__(
        self,
        root_dir: str | Path,
        *,
        virtual_mode: bool = True,
        timeout_sec: int = 90,
        work_dir: str = "/research_workspace/scratch",
        env_file: str | Path = ".env.code_runner.local",
    ) -> None:
        super().__init__(root_dir=root_dir, virtual_mode=virtual_mode, max_file_size_mb=20)
        self.timeout_sec = timeout_sec
        self.work_dir = work_dir
        self.exec_cwd = self._resolve_path(work_dir)
        self.exec_cwd.mkdir(parents=True, exist_ok=True)
        self.exec_env = self._build_exec_env(env_file)
        self._backend_id = f"guarded-local-exec:{self.cwd}"

    @property
    def id(self) -> str:
        return self._backend_id

    def _build_exec_env(self, env_file: str | Path) -> dict[str, str]:
        safe_env = {
            "PATH": os.environ.get("PATH", ""),
            "HOME": os.environ.get("HOME", ""),
            "PYTHONPATH": os.environ.get("PYTHONPATH", ""),
        }

        env_path = Path(env_file)
        if not env_path.is_absolute():
            env_path = self.cwd / env_path

        if not env_path.exists():
            return safe_env

        forbidden_markers = ("KEY", "TOKEN", "SECRET", "PASSWORD")
        for raw in env_path.read_text(encoding="utf-8").splitlines():
            line = raw.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('\"').strip("'")

            upper_key = key.upper()
            if any(marker in upper_key for marker in forbidden_markers):
                raise ValueError(
                    f"Forbidden secret-like key in {env_path}: {key}. Use non-secret env only."
                )

            safe_env[key] = value

        return safe_env

    def _validate_command(self, command: str) -> tuple[bool, str]:
        text = command.strip()
        if not text:
            return False, "Empty command"

        for token in self.BLOCKED_TOKENS:
            if token in text:
                return False, f"Blocked token detected: {token}"

        try:
            parts = shlex.split(text)
        except ValueError as e:
            return False, f"Invalid command parsing: {e}"

        if not parts:
            return False, "Command parse result is empty"

        if parts[0] not in self.ALLOWED_PREFIXES:
            allowed = ", ".join(sorted(self.ALLOWED_PREFIXES))
            return False, f"Command prefix '{parts[0]}' is not allowed. Allowed: {allowed}"

        return True, ""

    def execute(self, command: str) -> ExecuteResponse:
        ok, reason = self._validate_command(command)
        if not ok:
            return ExecuteResponse(output=f"Blocked command: {reason}", exit_code=126, truncated=False)

        args = shlex.split(command)

        try:
            completed = subprocess.run(
                args,
                shell=False,
                cwd=str(self.exec_cwd),
                env=self.exec_env,
                capture_output=True,
                text=True,
                timeout=self.timeout_sec,
            )
            output = (completed.stdout or "") + (completed.stderr or "")
            truncated = False
            if len(output) > 12_000:
                output = output[:12_000] + "\n...[truncated]"
                truncated = True

            return ExecuteResponse(output=output, exit_code=completed.returncode, truncated=truncated)
        except subprocess.TimeoutExpired:
            return ExecuteResponse(
                output=f"Command timed out after {self.timeout_sec}s",
                exit_code=124,
                truncated=False,
            )


In [5]:
model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

BASE_DIR = ROOT
RESEARCH_WORKSPACE_DIR = BASE_DIR / "research_workspace"
CODE_RUNNER_ENV_FILE = BASE_DIR / ".env.code_runner.local"
ACADEMIC_SKILL_SCRIPT = (BASE_DIR / "skills" / "academic-search" / "arxiv_search.py").resolve()

if not ACADEMIC_SKILL_SCRIPT.exists():
    raise FileNotFoundError(f"Skill script not found: {ACADEMIC_SKILL_SCRIPT}")

main_fs_backend = FilesystemBackend(root_dir=RESEARCH_WORKSPACE_DIR, virtual_mode=True, max_file_size_mb=20)

def main_backend_factory(rt: ToolRuntime):
    return CompositeBackend(default=StateBackend(rt), routes={"/": main_fs_backend})

code_runner_backend = GuardedLocalExecBackend(
    root_dir=BASE_DIR,
    virtual_mode=True,
    timeout_sec=90,
    work_dir="res_bc/",
    env_file=CODE_RUNNER_ENV_FILE,
)

def code_runner_backend_factory(rt: ToolRuntime):
    _ = rt
    return code_runner_backend


In [ ]:
code_runner_runnable = create_deep_agent(
    model=model,
    tools=[think_tool],
    system_prompt=(
        "You are code_runner. Use /research_workspace/scratch as your working directory. "
        "You can read skill scripts, create modified Python files, run them, and report command/exit_code/output."
    ),
    backend=code_runner_backend_factory,
)

code_runner_subagent = {
    "name": "code_runner",
    "description": "Run and verify Python code using local guarded execute backend.",
    "runnable": code_runner_runnable,
}

agent = create_deep_agent(
    model=model,
    tools=[think_tool],
    system_prompt="Use code_runner only for execution tasks.",
    backend=main_backend_factory,
    subagents=[code_runner_subagent],
)


In [ ]:
prompt = f"""
Use subagent_type='code_runner'.

1) Read this skill script: {ACADEMIC_SKILL_SCRIPT}
2) Create a modified copy at /research_workspace/scratch/arxiv_search_modified.py
   - keep same behavior
   - change default max_papers from 10 to 3
3) Run:
   python3 /research_workspace/scratch/arxiv_search_modified.py "large language model reasoning" --output-format markdown
4) Save the command output into /research_workspace/scratch/arxiv_results.md
5) Return command, exit code, and output file path
"""

result = agent.invoke({
    "messages": [
        {"role": "user", "content": prompt}
    ]
})

print(result["messages"][-1].content)


In [ ]:
output_path = Path("research_workspace/scratch/arxiv_results.md")
if output_path.exists():
    preview = output_path.read_text(encoding="utf-8")[:2000]
    print(preview)
else:
    print(f"Output file not found: {output_path}")